In [ ]:
!export TRANSFORMERS_NO_TORCHVISION=1
!pip -q install \
  "transformers==4.56.1" "accelerate>=0.33.0" "bitsandbytes>=0.43.3" \
  peft einops zstandard kernels



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# GPT-OSS + LoRA adapter — MXFP4-safe loader (no BitsAndBytes)
import os, io, tarfile, zstandard, json, torch, shutil
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # helps fragmentation

# 1) Register GPT-OSS BEFORE transformers auto classes
import kernels  # registers 'gpt_oss' arch

from pathlib import Path
from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, GenerationConfig

BASE_REPO   = "openai/gpt-oss-20b"     # base you fine-tuned
ADAPTER_TAR = "./ft-457d48d1-55b4_adapter-2025-09-06-00-39-48.tar.zst"
ADAPTER_DIR = "./adapter"
OFFLOAD_DIR = "./offload"
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

# 2) Extract adapter
with open(ADAPTER_TAR, "rb") as f:
    dctx = zstandard.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        with tarfile.open(fileobj=io.BufferedReader(reader), mode="r|") as tf:
            tf.extractall(ADAPTER_DIR)
print("Adapter files:", sorted(os.listdir(ADAPTER_DIR)))

# 3) Tokenizer (gets chat template)
tokenizer = AutoTokenizer.from_pretrained(BASE_REPO, use_fast=True, trust_remote_code=True)

# 4) Decide device map based on GPU capability (MXFP4 needs >= 7.5, e.g., T4/A40/A100)
use_cpu = True
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    if (major, minor) >= (7, 5):
        use_cpu = False
print("CUDA avail:", torch.cuda.is_available(), "| cap:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)

# For safety/headroom on 48GB (A40), leave a few GB free
device_map = "auto" if not use_cpu else {"": "cpu"}
max_memory = ({0: "45GiB", "cpu": "48GiB"} if not use_cpu else {"cpu": "48GiB"})

# 5) Load base — IMPORTANT:
#    - Do NOT pass BitsAndBytesConfig (bnb) with MXFP4.
#    - Do NOT pass torch_dtype here; it can trigger dequantization.
config = AutoConfig.from_pretrained(BASE_REPO, trust_remote_code=True)
print("quantization_config in base:", getattr(config, "quantization_config", None))

base = AutoModelForCausalLM.from_pretrained(
    BASE_REPO,
    config=config,
    trust_remote_code=True,
    device_map=device_map,
    max_memory=max_memory,
    low_cpu_mem_usage=True,
    offload_folder=OFFLOAD_DIR,   # enables disk offload if needed
)
print("✓ Loaded GPT-OSS base on", ("CPU" if use_cpu else "GPU"))
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# 6) Apply your LoRA adapter
model = PeftModel.from_pretrained(
    base,
    ADAPTER_DIR,
    device_map=device_map,
)
model.eval()
print("✓ Applied LoRA adapter")




Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json']


MXFP4 quantization requires triton >= 3.4.0 and kernels installed, we will default to dequantizing the model to bf16


CUDA avail: True | cap: (8, 0)
quantization_config in base: {'modules_to_not_convert': ['model.layers.*.self_attn', 'model.layers.*.mlp.router', 'model.embed_tokens', 'lm_head'], 'quant_method': 'mxfp4'}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Loaded GPT-OSS base on GPU
✓ Applied LoRA adapter


In [ ]:
# 7) Simple generation helper
def render_messages(messages):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        sys = next((m["content"] for m in messages if m["role"]=="system"), "")
        usr = next((m["content"] for m in messages if m["role"]=="user"), "")
        return f"<|system|>\n{sys}\n<|user|>\n{usr}\n<|assistant|>\n"

# Keep tokens modest at first; raise if memory allows
gen_cfg = GenerationConfig(
    max_new_tokens=256,
    temperature=0.7,       # not too low, avoids stuck loops
    top_p=0.9,
    repetition_penalty=1.15,  # pushes model to avoid repeats
    do_sample=True,
)

def generate(messages):
    prompt = render_messages(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, generation_config=gen_cfg)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip()
# 8) Provide PAPER CONTEXT safely:
# Paste your JSON array exactly between the triple quotes below.
paper_ctx_json = r'''
[
  {
    "sid": "title-0-0",
    "section": "Title",
    "text": "Grouplet: A Structured Image Representation for Recognizing Human and Object Interactions"
  },
  {
    "sid": "abstract-0-0",
    "section": "Abstract",
    "text": "Psychologists have proposed that many human-object interaction activities form unique classes of scenes."
  },
  {
    "sid": "abstract-0-1",
    "section": "Abstract",
    "text": "Recognizing these scenes is important for many social functions."
  },
  {
    "sid": "abstract-0-2",
    "section": "Abstract",
    "text": "To enable a computer to do this is however a challenging task."
  },
  {
    "sid": "abstract-0-3",
    "section": "Abstract",
    "text": "Take people-playing-musical-instrument (PPMI) as an example; to distinguish a person playing violin from a person just holding a violin requires subtle distinction of characteristic image features and feature arrangements that differentiate these two scenes."
  },
  {
    "sid": "abstract-0-4",
    "section": "Abstract",
    "text": "Most of the existing image representation methods are either too coarse (e.g. BoW) or too sparse (e.g. constellation models) for performing this task."
  },
  {
    "sid": "abstract-0-5",
    "section": "Abstract",
    "text": "In this paper, we propose a new image feature representation called \"grouplet\"."
  },
  {
    "sid": "abstract-0-6",
    "section": "Abstract",
    "text": "The grouplet captures the structured information of an image by encoding a number of discriminative visual features and their spatial configurations."
  },
  {
    "sid": "abstract-0-7",
    "section": "Abstract",
    "text": "Using a dataset of 7 different PPMI activities, we show that grouplets are more effective in classifying and detecting human-object interactions than other state-of-theart methods."
  },
  {
    "sid": "abstract-0-8",
    "section": "Abstract",
    "text": "In particular, our method can make a robust distinction between humans playing the instruments and humans co-occurring with the instruments without playing."
  },
  {
    "sid": "0-0-0",
    "section": "Introduction",
    "text": "In recent years, the computer vision field has made great progress in recognizing isolated objects, such as faces and cars."
  },
  {
    "sid": "0-0-1",
    "section": "Introduction",
    "text": "But a large proportion of our visual experience involves recognizing the interaction between objects."
  },
  {
    "sid": "0-0-2",
    "section": "Introduction",
    "text": "For example, seeing a human playing violin delivers a very different story than seeing a person chopping up a violin -one is a musician, the other is probably a contemporary artist."
  },
  {
    "sid": "0-0-3",
    "section": "Introduction",
    "text": "Psychologists have found that different brain areas are involved in recognizing different scenes of multiple objects <CIT> and in particular, there are neurons that react strongly upon seeing humans interacting with objects <CIT> ."
  },
  {
    "sid": "0-0-4",
    "section": "Introduction",
    "text": "Such evidence shows that the ability to recognize scenes of human-object interactions is fundamental to human cognition."
  },
  {
    "sid": "1-0-0",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Recognizing a person playing violin versus not playing violin requires subtle discriminations of image features."
  },
  {
    "sid": "1-0-1",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Our algorithm discovers discriminative features called grouplets that encode rich, structured information for such tasks."
  },
  {
    "sid": "1-0-2",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "In the left figure, three sample grouplets are shown in three different colors."
  },
  {
    "sid": "1-0-3",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Note that one grouplet (e.g. the cyan one) is represented by multiple image patches and their spatial configurations."
  },
  {
    "sid": "1-0-4",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "In the right figure, we show that some information of the grouplets from the left is missing (their hypothetical locations are indicated by dashed lines), prompting our algorithm to decide that the person is not playing violin."
  },
  {
    "sid": "1-1-0",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "The goal of our work is to use structured visual features to recognize scenes in which a person is interacting with a specific object in a specific manner, such as playing musical instruments."
  },
  {
    "sid": "1-1-1",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Humans can recognize such activities based on only static images, most likely due to the rich structured information in the activities."
  },
  {
    "sid": "1-1-2",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "For example, \"playing violin\" is defined not only by the appearance of a human and a violin and their co-occurrence, but also by the gesture of arms interacting with the pose of the violin, as shown in Fig. 1 ."
  },
  {
    "sid": "1-2-0",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "One intuitive approach for this problem is to design an algorithm that can recognize the human pose, the target object, and the spatial relationship between the human and the object <CIT> ."
  },
  {
    "sid": "1-2-1",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "It is, however, an exceedingly difficult task to recognize complex human gestures."
  },
  {
    "sid": "1-2-2",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Most of the human pose estimation algorithms today cannot reliably parse out body parts that are crucial to our task, especially with partial occlusions or in cluttered backgrounds <CIT> ."
  },
  {
    "sid": "1-2-3",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "The same is also true for object detection."
  },
  {
    "sid": "1-2-4",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Detection rates of generic objects in realistic scenes are still low <CIT> ."
  },
  {
    "sid": "1-3-0",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "In this paper, instead of exploring models for pose estimation or object detection, we approach the problem by discovering image features that can characterize well differ-ent human-object interactions."
  },
  {
    "sid": "1-3-1",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "We take the view in <CIT> that such human-object configurations are like different types of scenes."
  },
  {
    "sid": "1-3-2",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "So similar to scene and object classification [9, 18] , our features need to discover different classes of activities that carry intrinsically different visual appearance and spatial information."
  },
  {
    "sid": "1-3-3",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "This problem offers us an opportunity to explore the following issues that have not been widely studied in generic object recognition tasks: • Spatial relations among image patches."
  },
  {
    "sid": "1-3-4",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Recognizing that a person is playing violin is not simply discovering the co-occurrence of the violin and the human, which could also occur when a person just standing next to a violin."
  },
  {
    "sid": "1-3-5",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Our features need to capture the spatial relations that are crucial to define the human-object interactions. • More subtle and discriminative features."
  },
  {
    "sid": "1-3-6",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Most of the current image features (and models) are tested on classes of objects that are very different from each other (e.g. bicycles vs. cows)."
  },
  {
    "sid": "1-3-7",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "The classes of human-object interactions are much more similar, due to the dominant presence of humans in all classes."
  },
  {
    "sid": "1-3-8",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "This demands more discriminative features to encode the image differences."
  },
  {
    "sid": "1-3-9",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Focusing on the above issues, we propose a new image representation that encodes appearance, shape, and spatial relations of multiple image patches, termed \"grouplet\"."
  },
  {
    "sid": "1-3-10",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "The grouplets are discovered through a novel data mining approach, and could be further refined by a parameter estimation procedure."
  },
  {
    "sid": "1-3-11",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "We show that the methods using grouplets outperform the state-of-the-art approaches in both humanobject interaction classification and detection tasks."
  },
  {
    "sid": "1-4-0",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "The rest of this paper first presents a human-object interaction data set in Sec.2."
  },
  {
    "sid": "1-4-1",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Sec.3 and Sec.4 define the grouplets and introduce a method of obtaining discriminative grouplets respectively."
  },
  {
    "sid": "1-4-2",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Sec.5 briefly describes the classification methods that use grouplets."
  },
  {
    "sid": "1-4-3",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Related work is discussed in Sec.6."
  },
  {
    "sid": "1-4-4",
    "section": "1RW 3OD\\LQJ 9LROLQ 3OD\\LQJ 9LROLQ",
    "text": "Experiment results are reported in Sec.7."
  },
  {
    "sid": "2-0-0",
    "section": "The PPMI Dataset",
    "text": "Most of the popular image data sets are collected for recognizing generic objects [6, 5] or natural scenes <CIT> instead of human and object interactions."
  },
  {
    "sid": "2-0-1",
    "section": "The PPMI Dataset",
    "text": "We therefore collected a new data set called People-playing-musical-instruments (PPMI, Fig. 2 )."
  },
  {
    "sid": "2-0-2",
    "section": "The PPMI Dataset",
    "text": "PPMI 1 consists of 7 different musical instruments: bassoon, erhu, flute, French horn, guitar, saxophone, and violin."
  },
  {
    "sid": "2-0-3",
    "section": "The PPMI Dataset",
    "text": "Each class includes ∼150 PPMI+ images (humans playing instruments) and ∼150 PPMI-images (humans holding the instruments without playing)."
  },
  {
    "sid": "2-0-4",
    "section": "The PPMI Dataset",
    "text": "As Fig. 2 shows, images in PPMI are highly diverse and cluttered."
  },
  {
    "sid": "2-1-0",
    "section": "The PPMI Dataset",
    "text": "We focus on two problems on this data."
  },
  {
    "sid": "2-1-1",
    "section": "The PPMI Dataset",
    "text": "One is to classify different activities of humans playing instruments; the other is to distinguish PPMI+ and PPMI-images for each instru- 1 Resources of the images include image search engines Google, Yahoo, Baidu, and Bing, and photo hosting websites Flickr and Picassa.ment."
  },
  {
    "sid": "2-1-2",
    "section": "The PPMI Dataset",
    "text": "The latter task is very different from traditional image classification tasks."
  },
  {
    "sid": "2-1-3",
    "section": "The PPMI Dataset",
    "text": "Distinguishing PPMI+ and PPMI-images of the same instrument strongly depends on the structural information in the images, such as the spatial relations between the object and the human."
  },
  {
    "sid": "2-1-4",
    "section": "The PPMI Dataset",
    "text": "This property of our data set cannot be captured by <CIT> and <CIT> , which are possibly the only existing data sets of human-object interactions."
  },
  {
    "sid": "2-1-5",
    "section": "The PPMI Dataset",
    "text": "Besides classification, we also show results of detecting people playing different instruments on the PPMI dataset."
  },
  {
    "sid": "3-0-0",
    "section": "Image Building Block -the Grouplet",
    "text": "For recognizing human-object interactions, we discover a set of discriminative features that encode the structured image information."
  },
  {
    "sid": "3-0-1",
    "section": "Image Building Block -the Grouplet",
    "text": "To address the two central issues introduced in Sec.1, the grouplets have the following properties."
  },
  {
    "sid": "3-1-0",
    "section": "Image Building Block -the Grouplet",
    "text": "• Each grouplet contains a set of highly related image patches."
  },
  {
    "sid": "3-1-1",
    "section": "Image Building Block -the Grouplet",
    "text": "It encodes the appearance, location, and shape of these patches, as well as their spatial relationship. •"
  },
  {
    "sid": "3-1-2",
    "section": "Image Building Block -the Grouplet",
    "text": "For differentiating human and object interactions, we apply a novel data mining approach to discover a large number of discriminative grouplets."
  },
  {
    "sid": "3-1-3",
    "section": "Image Building Block -the Grouplet",
    "text": "A grouplet is defined by an AND/OR <CIT> structure on a set of feature units."
  },
  {
    "sid": "3-1-4",
    "section": "Image Building Block -the Grouplet",
    "text": "A feature unit, denoted by {A, x, σ}, indicates that a codeword of visual appearance A is observed in the neighborhood of location x (relative to a reference point)."
  },
  {
    "sid": "3-1-5",
    "section": "Image Building Block -the Grouplet",
    "text": "The spatial extent of A in the neighborhood of x is expressed as a 2D Gaussian distribution N (x, σ)."
  },
  {
    "sid": "3-2-0",
    "section": "Image Building Block -the Grouplet",
    "text": "Each ellipse denotes one feature unit."
  },
  {
    "sid": "3-2-1",
    "section": "Image Building Block -the Grouplet",
    "text": "A grouplet is formed by applying some OR operations and an AND operation to a set of feature units."
  },
  {
    "sid": "3-2-2",
    "section": "Image Building Block -the Grouplet",
    "text": "Each OR operation is applied to the feature units that have similar visual appearance and spatial extents, from which the one that has the strongest signal in the image is selected (thicker ellipses in Fig. 3 )."
  },
  {
    "sid": "3-2-3",
    "section": "Image Building Block -the Grouplet",
    "text": "The AND operation is applied to these selected feature units."
  },
  {
    "sid": "3-2-4",
    "section": "Image Building Block -the Grouplet",
    "text": "The size of a grouplet is the number of OR operations it contains."
  },
  {
    "sid": "3-3-0",
    "section": "Image Building Block -the Grouplet",
    "text": "In the grouplet representation, each feature unit captures a specific appearance, location, and spatial extent information of an image patch."
  },
  {
    "sid": "3-3-1",
    "section": "Image Building Block -the Grouplet",
    "text": "Together, the AND operation allows the grouplets to represent various interactions among a set of image patches, and the OR operation makes the grouplets resistent to small spatial variations."
  },
  {
    "sid": "3-3-2",
    "section": "Image Building Block -the Grouplet",
    "text": "By definition, we do not exert any constraint on the appearance or location of the feature units, nor the size of the grouplets."
  },
  {
    "sid": "3-3-3",
    "section": "Image Building Block -the Grouplet",
    "text": "Furthermore, the spatial extent of each feature unit will be automatically refined through a parameter estimation step (Sec.4.2.2), thus the grouplets can reflect any structured information among any number of image patches with any appearance."
  },
  {
    "sid": "3-3-4",
    "section": "Image Building Block -the Grouplet",
    "text": "Examples of grouplets are shown in Fig. 1 and Fig. 10 ."
  },
  {
    "sid": "3-4-0",
    "section": "Image Building Block -the Grouplet",
    "text": "Implementation Details: In the grouplet representation, SIFT descriptors <CIT> are computed over a dense image grid of D rectangular patches, as in <CIT> ."
  },
  {
    "sid": "3-4-1",
    "section": "Image Building Block -the Grouplet",
    "text": "Using k-means clustering, we obtain a SIFT codebook which contains 250 codewords."
  },
  {
    "sid": "3-4-2",
    "section": "Image Building Block -the Grouplet",
    "text": "Therefore, the visual appearance can be represented by {A w } W w=1 , where W =250."
  },
  {
    "sid": "3-4-3",
    "section": "Image Building Block -the Grouplet",
    "text": "The feature units in one OR operation should have the same visual codeword."
  },
  {
    "sid": "3-4-4",
    "section": "Image Building Block -the Grouplet",
    "text": "Reference points are chosen as the centers of the human faces."
  },
  {
    "sid": "4-0-0",
    "section": "Obtaining Discriminative Grouplets",
    "text": "To recognize subtly different scenes, we would like to find a rich set of grouplets that are not only highly char- acteristic of the image class, but also highly discriminative compared to other classes."
  },
  {
    "sid": "4-0-1",
    "section": "Obtaining Discriminative Grouplets",
    "text": "We propose a novel data mining algorithm for discovering discriminative grouplets."
  },
  {
    "sid": "5-0-0",
    "section": "Defining Discriminative Grouplets",
    "text": "Grouplet Λ is discriminative for class c means that Λ has strong signals on images of class c, and has weak signals on images of other classes."
  },
  {
    "sid": "5-0-1",
    "section": "Defining Discriminative Grouplets",
    "text": "In the rest of this section, we first describe how to compute the signal values of feature units and grouplets, and then elaborate on the definition of discriminative grouplets."
  },
  {
    "sid": "5-1-0",
    "section": "Defining Discriminative Grouplets",
    "text": "The signal v of a feature unit {A, x, σ} on an image I is the likelihood that {A, x, σ} is observed in I:"
  },
  {
    "sid": "5-2-0",
    "section": "Defining Discriminative Grouplets",
    "text": "EQUATION"
  },
  {
    "sid": "5-3-0",
    "section": "Defining Discriminative Grouplets",
    "text": "where Ω(x) is the image neighborhood of location x, a ′ is the appearance of the image patch at x ′ , p(A|a ′ ) is the probability that a ′ is assigned to codeword A. Please refer to Fig. 4 and implementation details of this section for more details."
  },
  {
    "sid": "5-3-1",
    "section": "Defining Discriminative Grouplets",
    "text": "For a codeword A w , we use a single variance σ w to encode its spatial distribution in all positions of the image."
  },
  {
    "sid": "5-4-0",
    "section": "Defining Discriminative Grouplets",
    "text": "Given the signal values of the feature units in a grouplet, each OR operation selects a feature unit that has the strongest signal (see Fig. 3 )."
  },
  {
    "sid": "5-4-1",
    "section": "Defining Discriminative Grouplets",
    "text": "The overall signal of the grouplet, i.e. result of the AND operation, is the smallest signal value of the selected feature units."
  },
  {
    "sid": "5-4-2",
    "section": "Defining Discriminative Grouplets",
    "text": "Intuitively, this decision ensures that even the relatively weakest feature unit needs to be strong enough for the grouplet to be strong (see Fig. 5 )."
  },
  {
    "sid": "5-4-3",
    "section": "Defining Discriminative Grouplets",
    "text": "In order to evaluate the discriminability of a grouplet, we introduce two terms, support value, Supp(⋅) and confidence value, Conf (⋅)."
  },
  {
    "sid": "5-4-4",
    "section": "Defining Discriminative Grouplets",
    "text": "A grouplet Λ is discriminative for a class c if both Supp(Λ, c) and Conf (Λ, c) are large."
  },
  {
    "sid": "5-4-5",
    "section": "Defining Discriminative Grouplets",
    "text": "Given a set of training images where the signal of Λ on image I i is denoted as r i , Supp(Λ, c) and Conf (Λ, c) are computed by where c i is the class label of I i ."
  },
  {
    "sid": "5-4-6",
    "section": "Defining Discriminative Grouplets",
    "text": "Intuitively, a large Supp(Λ, c) indicates that Λ generally has strong signals on images of class c, and a large Conf (Λ, c) implies relatively weak signals of Λ on images of classes other than c."
  },
  {
    "sid": "5-5-0",
    "section": "Defining Discriminative Grouplets",
    "text": "EQUATION"
  },
  {
    "sid": "5-6-0",
    "section": "Defining Discriminative Grouplets",
    "text": "Implementation Details: The size of Ω(x) is 5×5 patches."
  },
  {
    "sid": "5-6-1",
    "section": "Defining Discriminative Grouplets",
    "text": "We assign each image patch to its nearest codeword: p(A|a)=1 if and only if A is a's nearest codeword."
  },
  {
    "sid": "5-6-2",
    "section": "Defining Discriminative Grouplets",
    "text": "We initialize σ w to [0.6, 0; 0, 0.6] for any A w ."
  },
  {
    "sid": "5-6-3",
    "section": "Defining Discriminative Grouplets",
    "text": "σ w will be updated in the parameter estimation step (Sec.4.2.2)."
  },
  {
    "sid": "6-0-0",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "For each class, our goal is to find all the grouplets of large support and confidence values."
  },
  {
    "sid": "6-0-1",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "One way is to evaluate these values on all possible grouplets."
  },
  {
    "sid": "6-0-2",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "Assuming an image of D patches and a codeword vocabulary of size W , there are D×W possible feature units."
  },
  {
    "sid": "6-0-3",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "The total number of grouplets is therefore O(2 D×W ) (in this paper D×W =240250)."
  },
  {
    "sid": "6-0-4",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "Clearly, evaluating Supp(⋅) and Conf (⋅) of all the grouplets for each class is computationally infeasible."
  },
  {
    "sid": "6-1-0",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "We therefore develop a data mining algorithm for this task, which discriminatively explores the AND/OR structure of the grouplets in an Apriori mining <CIT> process."
  },
  {
    "sid": "6-1-1",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "Furthermore, we introduce a novel parameter estimation method to better estimate the spatial distribution σ w of each codeword A w as well as to obtain a set of weights for the grouplets of each class."
  },
  {
    "sid": "6-1-2",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "Our mining algorithm then iterates between the mining process and the parameter estimation process."
  },
  {
    "sid": "6-1-3",
    "section": "A Novel Iterative Mining Algorithm",
    "text": "An overview of the algorithm is shown in Algorithm 1, where l-grouplets indicate the grouplets of size l. We briefly describe the mining and the parameter estimation method in the rest of this section."
  },
  {
    "sid": "7-0-0",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "In each iteration of Algorithm 1, given the spatial distribution σ w of each codeword A w , we compute the signal values of all the feature units on each image as in Fig. 4 ."
  },
  {
    "sid": "7-0-1",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "We are then ready to mine the discriminative grouplets for every class."
  },
  {
    "sid": "7-1-0",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "We modify the Apriori <CIT> mining method to explore the AND/OR structures to select the discriminative grouplets."
  },
  {
    "sid": "7-1-1",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "The main idea of Apriori mining is compatible with the AND operation: if an l-grouplet has a large support value, then by removing the feature units in any of its OR opera- tions, the remaining (l-1)-grouplets also have large support values."
  },
  {
    "sid": "7-1-2",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "Therefore, we can generate l-grouplets based only on the mined (l -1)-grouplets, instead of considering all the possibilities."
  },
  {
    "sid": "7-1-3",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "The OR operation is used to obtain the 1grouplets."
  },
  {
    "sid": "7-1-4",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "For each codeword, a hierarchical clustering is applied to the feature units that have large enough support values."
  },
  {
    "sid": "7-1-5",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "Each cluster is then initialized as a 1-grouplet."
  },
  {
    "sid": "7-1-6",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "The mining process is briefly shown in Algorithm 1."
  },
  {
    "sid": "7-2-0",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "Implementation Details: The hierarchical clustering is based on the maximum distance metric, of which the threshold is two times the patch size."
  },
  {
    "sid": "7-2-1",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "The mining algorithm automatically adjusts the values of T Supp and T Conf for each class, so that the number of mined grouplets for different classes are approximately the same."
  },
  {
    "sid": "7-2-2",
    "section": "The Modified Apriori Mining Algorithm",
    "text": "More details of the mining method can be found in the supplementary document of this paper."
  },
  {
    "sid": "8-0-0",
    "section": "Refining Grouplets",
    "text": "Given a set of mined grouplets, we introduce a parameter estimation method to further refine the spatial distribution σ w of each codeword A w ."
  },
  {
    "sid": "8-0-1",
    "section": "Refining Grouplets",
    "text": "With the refined σ, one can expect that more accurate signal values of the feature units can be computed, which in turn can be put into the mining process to obtain better grouplets in the next iteration."
  },
  {
    "sid": "8-0-2",
    "section": "Refining Grouplets",
    "text": "Furthermore, the algorithm computes a weight on each mined grouplet for each class."
  },
  {
    "sid": "8-0-3",
    "section": "Refining Grouplets",
    "text": "The combination of grouplets and the class-dependent weights can then be directly used for classification tasks (see Sec.5)."
  },
  {
    "sid": "8-1-0",
    "section": "Refining Grouplets",
    "text": "Given an image I with class label c, we compute the likelihood of I given a set of parameters θ, where θ contains the parameters for the spatial extent of each codeword and the importance of each grouplet."
  },
  {
    "sid": "9-0-0",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "∑"
  },
  {
    "sid": "9-1-0",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "EQUATION"
  },
  {
    "sid": "9-2-0",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "where Λ m indicates the m-th mined grouplet."
  },
  {
    "sid": "9-2-1",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "p(I|Λ m , θ) denotes the likelihood of I given Λ m ."
  },
  {
    "sid": "9-2-2",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "p(Λ m |c, θ) models the importance of Λ m for class c. We use an expectationmaximization (EM) algorithm to estimate the parameters θ."
  },
  {
    "sid": "9-3-0",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "Due to space limitation, we elaborate the details of the parameter estimation method in the supplementary document."
  },
  {
    "sid": "9-3-1",
    "section": "p(I, c|θ) = p(c|θ)",
    "text": "On a PC with a 2.66GHz CPU, our algorithm can process around 20000 grouplets under 3 minutes per EM iteration."
  },
  {
    "sid": "10-0-0",
    "section": "Using Grouplets for Classification",
    "text": "Having obtained the discriminative grouplets, we are ready to use them for classification tasks."
  },
  {
    "sid": "10-0-1",
    "section": "Using Grouplets for Classification",
    "text": "In this paper, we show that grouplets can be used for classification either by a generative or a discriminative classifier."
  },
  {
    "sid": "10-1-0",
    "section": "Using Grouplets for Classification",
    "text": "A Generative Classifier."
  },
  {
    "sid": "10-1-1",
    "section": "Using Grouplets for Classification",
    "text": "Recall that in Sec.4.2.2, our probabilistic parameter estimation process outputs the importance of each grouplet for each class."
  },
  {
    "sid": "10-1-2",
    "section": "Using Grouplets for Classification",
    "text": "This can, therefore, be directly used for classification."
  },
  {
    "sid": "10-1-3",
    "section": "Using Grouplets for Classification",
    "text": "Given a new image I, its class label c is predicted as follows,"
  },
  {
    "sid": "10-2-0",
    "section": "Using Grouplets for Classification",
    "text": "EQUATION"
  },
  {
    "sid": "10-3-0",
    "section": "Using Grouplets for Classification",
    "text": "A Discriminative Classifier."
  },
  {
    "sid": "10-3-1",
    "section": "Using Grouplets for Classification",
    "text": "Discriminative classifiers such as SVM can be applied by using groupets."
  },
  {
    "sid": "10-3-2",
    "section": "Using Grouplets for Classification",
    "text": "Given an image, the input feature vector to SVM classifiers is the signal values of the mined grouplets."
  },
  {
    "sid": "11-0-0",
    "section": "Related Work",
    "text": "Many features have been proposed for various vision tasks in the past decade <CIT> ."
  },
  {
    "sid": "11-0-1",
    "section": "Related Work",
    "text": "It is out of the scope of this paper to discuss all of them."
  },
  {
    "sid": "11-0-2",
    "section": "Related Work",
    "text": "Instead, we discuss the image representations that have directly influenced our work."
  },
  {
    "sid": "11-1-0",
    "section": "Related Work",
    "text": "One of the most popular image feature representation schemes is bag of words (BoW) and its derivations (e.g. <CIT> )."
  },
  {
    "sid": "11-1-1",
    "section": "Related Work",
    "text": "These methods have shown promising results in holistic image classification tasks."
  },
  {
    "sid": "11-1-2",
    "section": "Related Work",
    "text": "But by assuming little or no spatial relationships among image patches, these representations are not sufficient for more demanding tasks such as differentiating human and object interactions."
  },
  {
    "sid": "11-2-0",
    "section": "Related Work",
    "text": "In order to remedy BoW, some methods have been proposed to either encode longer range image statistics [2 <CIT> ] or explicitly model spatial relationships among image patches [9, 7, 20] ."
  },
  {
    "sid": "11-2-1",
    "section": "Related Work",
    "text": "But most of such approaches uncover image features in a generative way, which might result in some features that are not essential for recognition."
  },
  {
    "sid": "11-2-2",
    "section": "Related Work",
    "text": "In <CIT> , a deformable part model is presented for discriminatively detecting objects in cluttered scenes."
  },
  {
    "sid": "11-2-3",
    "section": "Related Work",
    "text": "This method, however, assumes that the target object consists of a small number of deformable parts, which might not be able to model the subtle difference between similar image categories."
  },
  {
    "sid": "11-3-0",
    "section": "Related Work",
    "text": "Our feature is similar in spirit to <CIT> , though independently developed."
  },
  {
    "sid": "11-3-1",
    "section": "Related Work",
    "text": "We differ from <CIT> in that our features are automatically discovered instead of supervised by humans, making it a more scalable and convenient algorithm."
  },
  {
    "sid": "11-3-2",
    "section": "Related Work",
    "text": "Furthermore, we emphasize the dependence among image features, which is critical for demanding recognition tasks such as human and object interactions."
  },
  {
    "sid": "11-4-0",
    "section": "Related Work",
    "text": "There has been a lot of work on discriminative feature selection [1 <CIT> ] ."
  },
  {
    "sid": "11-4-1",
    "section": "Related Work",
    "text": "But most of the methods are not able to manage such a huge number of features (2 to the power of millions) as in the grouplets."
  },
  {
    "sid": "11-4-2",
    "section": "Related Work",
    "text": "Our algorithm is inspired by previous works [2 <CIT> ] that also use data mining methods for feature selection."
  },
  {
    "sid": "11-4-3",
    "section": "Related Work",
    "text": "But compared to these previous methods, we take a step further to encode much more structured information in the feature representation."
  },
  {
    "sid": "12-0-0",
    "section": "Experiment",
    "text": "We first conduct experiments to analyze the properties of grouplets (Sec.7.1)."
  },
  {
    "sid": "12-0-1",
    "section": "Experiment",
    "text": "The rest of this section then focuses on comparing using grouplets for human-object interaction classification and detection with a number of existing stateof-the-art methods."
  },
  {
    "sid": "12-0-2",
    "section": "Experiment",
    "text": "Apart from Sec.7.5, all experiments use the PPMI dataset introduced in Sec.2."
  },
  {
    "sid": "12-0-3",
    "section": "Experiment",
    "text": "In Sec.7.4 we use the original PPMI images."
  },
  {
    "sid": "12-0-4",
    "section": "Experiment",
    "text": "Data sets that are used from Sec.7.1 to 7.3 are obtained as follows."
  },
  {
    "sid": "12-0-5",
    "section": "Experiment",
    "text": "We first run a face detector <CIT> on all PPMI images."
  },
  {
    "sid": "12-0-6",
    "section": "Experiment",
    "text": "For each instrument, we manually select 200 detection results from PPMI+ and PPMIimages respectively."
  },
  {
    "sid": "12-0-7",
    "section": "Experiment",
    "text": "We then crop a rectangle region of the upper body of each selected detection result and normalize the region to 256×256 pixels so that the face size is 32×32."
  },
  {
    "sid": "13-0-0",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "Effects of the grouplet size We use a 7-class classification task to analyze the properties of the mined grouplets (experiment details in Sec.7.2)."
  },
  {
    "sid": "13-0-1",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "Here we use an SVM with the histogram intersection kernel for classification."
  },
  {
    "sid": "13-1-0",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "Because the AND operation takes the smallest signal value of all feature units, it is unlikely that grouplets with a very large size can be mined."
  },
  {
    "sid": "13-1-1",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "We observe that a majority of the mined grouplets contain <CIT> , or 3 feature units."
  },
  {
    "sid": "13-1-2",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "We see a big increase in accuracy using grouplets from size 1 to size 3."
  },
  {
    "sid": "13-1-3",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "After this, the accuracy stabilizes even when including grouplets of bigger sizes."
  },
  {
    "sid": "13-1-4",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "Two reasons might account for this observation: 1) the number of grouplets containing more than 3 feature units is small, and hence the overall contribution to classification is small; 2) much information in such grouplets is already contained in the grouplets of smaller sizes."
  },
  {
    "sid": "13-1-5",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "Effect of the Iterative Learning Procedure Given a set of training images, our algorithm iterates between a mining and a parameter estimation step."
  },
  {
    "sid": "13-1-6",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "The idea is that each iteration offers a better refinement of the grouplet parameters (e.g. spatial extent of codeword), hence of the overall dscriminability."
  },
  {
    "sid": "13-1-7",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "We observe the biggest gain between the first and the second iteration, indicating that with only two iterations, the method can obtain a good estimation of the spatial extent of each grouplet."
  },
  {
    "sid": "13-2-0",
    "section": "Analysis of the Properties of the Grouplets",
    "text": "*URXSOHW VL]H 1R RI PLQHG *URXSOHWV 0D[LPXP VL]H RI JURXSOHWV &ODVVLILFDWLRQ DFFXUDF\\"
  },
  {
    "sid": "14-0-0",
    "section": "Classification of Playing Different Instruments",
    "text": "Here we use our algorithm (grouplet+SVM and grou-plet+Model, Sec.5) to classify images of people playing seven different musical instruments."
  },
  {
    "sid": "14-0-1",
    "section": "Classification of Playing Different Instruments",
    "text": "For each class, 100 normalized PPMI+ images are randomly selected for training and the remaining 100 images for testing."
  },
  {
    "sid": "14-0-2",
    "section": "Classification of Playing Different Instruments",
    "text": "We use three iterations of the iterative learning framework to mine around 2000 grouplets for each class."
  },
  {
    "sid": "14-0-3",
    "section": "Classification of Playing Different Instruments",
    "text": "We observe that the histogram intersection kernel performs better than the other kernels."
  },
  {
    "sid": "14-1-0",
    "section": "Classification of Playing Different Instruments",
    "text": "We compare our method with some other approaches."
  },
  {
    "sid": "14-1-1",
    "section": "Classification of Playing Different Instruments",
    "text": "The results are shown in Fig. 8 (right)."
  },
  {
    "sid": "14-1-2",
    "section": "Classification of Playing Different Instruments",
    "text": "Both BoW and SPM <CIT> use the histogram representation, where BoW does not consider spatial information in image features while SPM accounts for some level of coarse spatial information by building histograms in different regions of the image."
  },
  {
    "sid": "14-1-3",
    "section": "Classification of Playing Different Instruments",
    "text": "The BoW representation is followed by an SVM classifier with the histogram intersection kernel."
  },
  {
    "sid": "14-1-4",
    "section": "Classification of Playing Different Instruments",
    "text": "Both DPM <CIT> and the constellation model <CIT> are part-based models, where DPM trains the classifier discriminatively and constellation model adopts a generative way."
  },
  {
    "sid": "14-2-0",
    "section": "Classification of Playing Different Instruments",
    "text": "We observe that our grouplet+SVM outperforms the other methods by a large margin."
  },
  {
    "sid": "14-2-1",
    "section": "Classification of Playing Different Instruments",
    "text": "This suggests the effectiveness of the structural information in the mined grou-"
  },
  {
    "sid": "14-3-0",
    "section": "Classification of Playing Different Instruments",
    "text": "%DVVRRQ (UKX )OXWH )UHQFK +RUQ *XLWDU 6D[RSKRQH 9LROLQ % D V V R R Q ( UK X ) OX WH ) UH Q F K + R UQ * X LW D U 6 D [ R S K R Q H 9 LR"
  },
  {
    "sid": "14-4-0",
    "section": "Classification of Playing Different Instruments",
    "text": "OLQ *URXSOHW 690 *URXSOHW 0RGHO 630 '30 &RQVWHO ODWLRQ %R: Confusion matrix obtained by grouplet+SVM."
  },
  {
    "sid": "14-4-1",
    "section": "Classification of Playing Different Instruments",
    "text": "The classification accuracy is 65.7%, whereas chance is 14%."
  },
  {
    "sid": "14-4-2",
    "section": "Classification of Playing Different Instruments",
    "text": "right: Classification results of different methods: our grouplet+SVM, our grouplet+Model, a four-level spatial pyramid matching (SPM) <CIT> , the deformable part model (DPM) <CIT> , the constellation model <CIT> , and bag-of-words (BoW)."
  },
  {
    "sid": "14-4-3",
    "section": "Classification of Playing Different Instruments",
    "text": "Y-axis indicates the classification accuracy of each method on the 7 classes."
  },
  {
    "sid": "14-5-0",
    "section": "Classification of Playing Different Instruments",
    "text": "plets."
  },
  {
    "sid": "14-5-1",
    "section": "Classification of Playing Different Instruments",
    "text": "Furthermore, the method that combines grouplet with a generative model achieves comparable performance with SPM."
  },
  {
    "sid": "14-5-2",
    "section": "Classification of Playing Different Instruments",
    "text": "This demonstrates that (1) the discriminatively mined grouplets carry the information that can distinguish images of different classes; (2) our parameter estimation step can effectively learn the weights of each mined grouplet."
  },
  {
    "sid": "15-0-0",
    "section": "Discriminating Playing from Not Playing",
    "text": "Our algorithm aims to learn discriminative structured information of human-object interactions."
  },
  {
    "sid": "15-0-1",
    "section": "Discriminating Playing from Not Playing",
    "text": "To demonstrate this, we conduct a classification experiment on PPMI+ vs. PPMI-datasets."
  },
  {
    "sid": "15-0-2",
    "section": "Discriminating Playing from Not Playing",
    "text": "For each instrument, we perform a binary classification task: whether the picture contains a person playing the instrument or a person not playing the instrument."
  },
  {
    "sid": "15-0-3",
    "section": "Discriminating Playing from Not Playing",
    "text": "Note that all images contain person(s) and instrument(s)."
  },
  {
    "sid": "15-0-4",
    "section": "Discriminating Playing from Not Playing",
    "text": "The distinction between PPMI+ and PPMI-is only the way the person is interacting with the instrument."
  },
  {
    "sid": "15-1-0",
    "section": "Discriminating Playing from Not Playing",
    "text": "We have 7 binary classification problems."
  },
  {
    "sid": "15-1-1",
    "section": "Discriminating Playing from Not Playing",
    "text": "In each problem, 100 normalized PPMI+ and 100 PPMI-images are randomly selected for training, and the other 200 images are used for testing."
  },
  {
    "sid": "15-1-2",
    "section": "Discriminating Playing from Not Playing",
    "text": "We mine around 4000 grouplets for both PPMI+ and PPMI-images of each instrument."
  },
  {
    "sid": "15-1-3",
    "section": "Discriminating Playing from Not Playing",
    "text": "In Table 1, our method is compared with the other approaches described in Sec. 7 not listed in Table 1 ."
  },
  {
    "sid": "15-1-4",
    "section": "Discriminating Playing from Not Playing",
    "text": "We can see that our method outperforms the other methods on almost all the classes, especially on bassoon, flute, and violin, where our approach improves the accuracy by almost 10%."
  },
  {
    "sid": "15-1-5",
    "section": "Discriminating Playing from Not Playing",
    "text": "The only exception is guitar, where DPM achieves the best performance."
  },
  {
    "sid": "15-1-6",
    "section": "Discriminating Playing from Not Playing",
    "text": "The reason is that in the normalized images of people playing guitar, the guitar always occupies a big region at the left-bottom part of the image (Fig. 10 )."
  },
  {
    "sid": "15-1-7",
    "section": "Discriminating Playing from Not Playing",
    "text": "Therefore it is not difficult for the part-based methods (DPM, SPM) to localize the guitar in each image."
  },
  {
    "sid": "15-1-8",
    "section": "Discriminating Playing from Not Playing",
    "text": "Compared with Fig. 10(d) , much fewer grouplets are observed on PPMI-images."
  },
  {
    "sid": "16-0-0",
    "section": "Detecting Human and Object Interactions",
    "text": "Here, we test our approach's ability to detect activities in cluttered scenes."
  },
  {
    "sid": "16-0-1",
    "section": "Detecting Human and Object Interactions",
    "text": "We use the original PPMI images as shown in Fig. 2 ."
  },
  {
    "sid": "16-0-2",
    "section": "Detecting Human and Object Interactions",
    "text": "In this experiment, 80 PPMI+ and 80 PPMI-randomly selected images of each instrument are used for training, and the remaining images for testing."
  },
  {
    "sid": "16-1-0",
    "section": "Detecting Human and Object Interactions",
    "text": "We first run a face detector on all images."
  },
  {
    "sid": "16-1-1",
    "section": "Detecting Human and Object Interactions",
    "text": "We set a relatively low detection threshold to guarantee that almost all human faces are detected."
  },
  {
    "sid": "16-1-2",
    "section": "Detecting Human and Object Interactions",
    "text": "Given each face detection, we crop out the neighboring region."
  },
  {
    "sid": "16-1-3",
    "section": "Detecting Human and Object Interactions",
    "text": "Based on these regions, we mine the grouplets that are discriminative for detecting people playing each instrument."
  },
  {
    "sid": "16-1-4",
    "section": "Detecting Human and Object Interactions",
    "text": "Then, an 8-class SVM classifier is trained to determine whether this detection contains a person playing one of the 7 instruments or not."
  },
  {
    "sid": "16-1-5",
    "section": "Detecting Human and Object Interactions",
    "text": "This is a very challenging task (see Fig. 9 )."
  },
  {
    "sid": "16-1-6",
    "section": "Detecting Human and Object Interactions",
    "text": "The preliminary experiment result shows that, measured with area under the precision-recall curve, our algorithm significantly outperforms the SPM method <CIT> : we obtain a 45.7% performance, while SPM is 37.3%."
  },
  {
    "sid": "16-1-7",
    "section": "Detecting Human and Object Interactions",
    "text": "We show examples of both successes and failures of our algorithm and SPM in Fig. 9 , from which we can see that SPM produces more false alarms than our method."
  },
  {
    "sid": "17-0-0",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "Not only grouplets can be used for recognizing humanobject interactions, but it is also a general framework to mine structured visual features in images."
  },
  {
    "sid": "17-0-1",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "Therefore we also test our algorithm in an object recognition task using Caltech101 <CIT> , in the same setting as in <CIT> ."
  },
  {
    "sid": "17-0-2",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "Other than the method in <CIT> , our model performs on par with most of the state-of-the-art algorithms."
  },
  {
    "sid": "17-0-3",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "It is important to note that this experiment is carried out without any additional tuning of the algorithm designed activity classification."
  },
  {
    "sid": "17-0-4",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "To objects that are not characterized by specific spatial structures (e.g. articulated animals), some design modifications should be applied to mine the grouplets."
  },
  {
    "sid": "17-0-5",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "Method <CIT> Grouplet+SVM Accuracy 48% 59% 65% 77% 62%"
  },
  {
    "sid": "17-1-0",
    "section": "Result on Other Dataset -Caltech 101",
    "text": "The performance is measured by the average accuracy of the 101 classes."
  },
  {
    "sid": "18-0-0",
    "section": "Conclusion",
    "text": "In this work, we proposed a grouplet feature for recognizing human-object interactions."
  },
  {
    "sid": "18-0-1",
    "section": "Conclusion",
    "text": "Grouplets encode detailed and structured information in the image data."
  },
  {
    "sid": "18-0-2",
    "section": "Conclusion",
    "text": "A data mining method incorporated with a parameter estimation step is applied to mine the discriminative grouplets."
  },
  {
    "sid": "18-0-3",
    "section": "Conclusion",
    "text": "One future research direction would be to link the mined grouplets with semantic meanings in the images to obtain deeper understanding of the scenes of human-object interactions."
  }
]
'''.strip()

# Try to parse JSON and convert to a compact plain-text context
try:
    data = json.loads(paper_ctx_json)
    # Join first N entries to keep prompt size reasonable
    N = 200
    parts = []
    for it in data[:N]:
        sec = it.get("section", "")
        txt = it.get("text", "")
        if sec and txt:
            parts.append(f"{sec}: {txt}")
        elif txt:
            parts.append(txt)
    paper_ctx = "\n".join(parts)
except Exception as e:
    # Fallback: use the raw string if parsing fails
    paper_ctx = paper_ctx_json

# 9) Ask the model — enforce + repair JSON

# A) Safer generation config that the model won't override
gen_cfg = GenerationConfig(
    max_new_tokens=1200,   # more room
    do_sample=False,       # deterministic; good for strict formats
    temperature=None,      # ensure not set
    top_p=None,
    repetition_penalty=1.02,
    pad_token_id=tokenizer.pad_token_id or 199999,
    bos_token_id=tokenizer.bos_token_id or 199998,
    eos_token_id=tokenizer.eos_token_id or 200002,
)

# B) Trim the paper context harder (e.g., first 120 items)
N = 120
parts = []
for it in data[:N]:
    sec = it.get("section","")
    txt = it.get("text","")
    if sec and txt:
        parts.append(f"{sec}: {txt}")
paper_ctx = "\n".join(parts)

# C) Stronger system message + concrete JSON example (NO ellipses)
sys_msg = (
  "You are an assistant that outputs ONLY valid JSON. "
  "Rules: use DOUBLE QUOTES; no trailing commas; no comments; no extra text; "
  "DO NOT output the literal string \"...\" anywhere; every slide must have non-empty title and bullets.\n\n"
  "Return an object like:\n"
  "{\n"
  "  \"slide 1\": { \"text\": { \"title\": \"Problem & Motivation\", \"bullets\": [\"Why HOI recognition matters\", \"Limitations of BoW/constellation\"] } },\n"
  "  \"slide 2\": { \"text\": { \"title\": \"Proposed: Grouplets\", \"bullets\": [\"Encode appearance+location+shape\", \"AND/OR structure across patches\"] } }\n"
  "}\n"
)

msgs = [
  {"role": "system", "content": sys_msg},
  {"role": "user", "content":
    "Generate academic slides (6–8 slides) from this paper. "
    "Use concise factual titles and short bullet points (3–5 per slide). "
    "Output ONLY JSON as described.\n\n=== PAPER CONTEXT ===\n" + paper_ctx}
]

prompt = render_messages(msgs)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, generation_config=gen_cfg, use_model_defaults=False, return_dict_in_generate=True)

gen_ids = out.sequences[0, inputs["input_ids"].shape[-1]:]   # only new tokens
raw = tokenizer.decode(gen_ids, skip_special_tokens=True)


import json, re

def strip_code_fences(s: str) -> str:
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", s, re.IGNORECASE)
    return m.group(1) if m else s

def first_balanced_json(s: str) -> str:
    s = s.strip()
    start = s.find("{")
    if start == -1:
        raise ValueError("No JSON object start '{' found")
    in_str = False
    esc = False
    depth = 0
    for i, ch in enumerate(s[start:], start=start):
        if ch == "\\" and not esc:
            esc = True
            continue
        if ch == '"' and not esc:
            in_str = not in_str
        if not in_str:
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start:i+1]
        esc = False
    raise ValueError("No balanced JSON found")

def fix_trailing_commas(s: str) -> str:
    s = re.sub(r",\s*([}\]])", r"\1", s)  # ,} or ,]
    return s

def sanitize_json_text(s: str) -> str:
    s = strip_code_fences(s)
    s = first_balanced_json(s)
    s = fix_trailing_commas(s)
    return s

json_str = sanitize_json_text(raw)
slides = json.loads(json_str)
print(json.dumps(slides, indent=2))




{
  "slide 1": {
    "text": {
      "title": "Human-Object Interaction Recognition",
      "bullets": [
        "Recognizing human-object interactions is fundamental to human cognition",
        "Humans can recognize such activities based on only static images",
        "Most of the existing image representation methods are either too coarse or too sparse for performing this task"
      ]
    }
  },
  "slide 2": {
    "text": {
      "title": "People Playing Musical Instruments (PPMI)",
      "bullets": [
        "PPMI 1 consists of 7 different musical instruments: bassoon, erhu, flute, French horn, guitar, saxophone, and violin",
        "Each class includes ~150 PPMI+ images (humans playing instruments) and ~150 PPMI-images (humans holding the instruments without playing)",
        "Images in PPMI are highly diverse and cluttered"
      ]
    }
  },
  "slide 3": {
    "text": {
      "title": "Grouplet Representation",
      "bullets": [
        "Each grouplet contains a set of high

In [ ]:
!nvidia-smi

Sat Sep  6 03:34:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.05             Driver Version: 550.127.05     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:61:00.0 Off |                    0 |
| N/A   35C    P0             68W /  300W |   59619MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
